# 12_02 Instructions and prompts: how does a text predictor end up taking turns?

SmolLM2 was pretrained to continue text, like every language model, and then **instruction-tuned** on
conversations. In this notebook you feed it the same exchange as raw text and through its chat template,
look inside the template, and then use a prompt to route Kittiwake's support tickets, measured against
the classifier you trained in Lab 03.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-12-large-and-small-language-models", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json, time
import torch
import slm
from nlpcheck import ask, guess, reveal, check_12_02

t = time.time()
tok, model = slm.load()          # SmolLM2-360M-Instruct, from the image; nothing downloads
print(f"model loaded in {time.time() - t:.0f} s: {sum(p.numel() for p in model.parameters()):,} weights, "
      f"a vocabulary of {len(tok):,} tokens")

## 1. Recall

**r3.** What does temperature 0 mean? (a) always the most likely token, (b) a random token,
(c) the least likely token

**r4.** What is a logit? (a) a probability between 0 and 1, (b) the raw score the last layer gives a
token, before softmax, (c) a token id

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. Raw text against the chat template

First the exchange as plain text, the way a transcript would look in the model's training data. Will the
model write the agent's reply and stop? Guess `"yes"` or `"no"`. The two generations take about a
minute.

In [ ]:
guess("raw_stops", None)   # "yes" or "no"

In [ ]:
raw = "Customer: My phone won't turn on.\nAgent:"
raw_reply = slm.generate(raw, max_new_tokens=40)
print("RAW TEXT:\n" + raw + raw_reply)

chat = [{"role": "user", "content": "My phone won't turn on."}]
chat_reply = slm.generate(chat, max_new_tokens=40)
print("\nCHAT TEMPLATE:\n" + chat_reply)
reveal("raw_stops", "no" if "Customer:" in raw_reply else "yes")

No. As raw text the model writes the agent's line and then the customer's next line, and would go on
writing both sides until it ran out of tokens, because a transcript continues. Nothing in the text says
whose turn the model is.

Through the chat template the same weights give one reply and stop. The difference is not a different
model; it is text the model was trained on. After pretraining on trillions of tokens, SmolLM2 was
fine-tuned on conversations written in one fixed format (**supervised fine-tuning**, SFT), then tuned
again towards the replies people preferred (**preference tuning**; SmolLM2 used DPO). So after the
marker that opens an assistant turn the likely next tokens are a reply, and after the reply the likely
next token is the marker that ends it, which `generate()` treats as the stop token.

## 3. Inside the template

`apply_chat_template` writes the conversation out as the exact text the model reads. A 7-token question
goes in: how many tokens does the template add? Guess a number.

In [ ]:
guess("template_tokens", None)   # how many tokens the template adds to the question

In [ ]:
q = "What is the capital of France?"
wrapped = slm.as_text([{"role": "user", "content": q}])
print(wrapped)
n_q = len(tok(q, add_special_tokens=False).input_ids)
n_all = len(tok(wrapped, add_special_tokens=False).input_ids)
template_tokens = n_all - n_q
print(f"question {n_q} tokens, as the model reads it {n_all} tokens")
reveal("template_tokens", template_tokens)

Thirty. Most of them are a **system message** you never wrote: "You are a helpful AI assistant named
SmolLM, trained by Hugging Face". The template inserts it when the conversation has none, which is why the
model sometimes opens a reply with "Hugging Face". A system message is only more text at the front: it
changes the probabilities of what follows, and it is where an application puts its instructions.

## 4. Can a prompt route tickets?

Lab 03 trained a classifier on 500 labelled tickets. A language model needs no training: describe the four
departments in a system message, show it the ticket, start its reply with `Department:`, and read the
logits of the four department names as the next token. One forward pass per ticket, no text generated.
The worked example routes one ticket:

In [ ]:
train, test = slm.tickets()
sample = slm.routing_sample()       # 12 held-out tickets, the same for everyone
print(slm.ROUTING_PROMPT, "\n")
t = sample.text[0]
scores = slm.department_logits(t)
print(t)
print(dict(zip(slm.DEPARTMENTS, scores)), "->", slm.DEPARTMENTS[scores.index(max(scores))],
      "| label:", sample.department[0])

Now all 12. The session computed these at start-up in the background; if that has not finished, the cell
does it now, which takes about a minute and a half. How many of the 12 will the prompt route correctly? Lab 03's
classifier gets all 12.

In [ ]:
guess("prompt_routing_right", None)   # how many of 12

In [ ]:
L = slm.routing_logits(list(sample.text))
labels = list(sample.department)
raw_pred = [slm.DEPARTMENTS[r.index(max(r))] for r in L]
right = sum(p == y for p, y in zip(raw_pred, labels))
for p, y in zip(raw_pred, labels):
    print(f"  predicted {p:8} label {y}")
print("predicted counts:", {d: raw_pred.count(d) for d in slm.DEPARTMENTS})
reveal("prompt_routing_right", right)

Look at the counts before the score: most tickets were sent to `billing`, whatever they said. The model
has a **prior**: some department names are simply more likely after "Department:" than others, and that
preference is added to every ticket's evidence.

## 5. Your turn: calibrate

If the bias is the same for every ticket, it can be measured and removed. Subtract each department's
average logit over the 12 tickets (the mean of each column), then take the argmax again. This is a
simple form of **calibration**; it needs no new forward passes.

In [ ]:
Lt = torch.tensor(L)
calibrated = None   # YOUR CODE HERE: Lt minus the mean of each column, Lt.mean(0)
cal_pred = [slm.DEPARTMENTS[int(i)] for i in calibrated.argmax(1)] if calibrated is not None else None
if cal_pred:
    print("calibrated:", sum(p == y for p, y in zip(cal_pred, labels)), "of 12 right;",
          {d: cal_pred.count(d) for d in slm.DEPARTMENTS})

## 6. The classifier, for comparison

Lab 03's pipeline, trained on the 500 labelled tickets, scored on the 100 held out.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
t = time.time()
clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                    LogisticRegression(max_iter=1000)).fit(train.text, train.department)
fit_s = time.time() - t
t = time.time()
acc = float((clf.predict(test.text) == test.department).mean())
per = (time.time() - t) / len(test)
print(f"trained in {fit_s:.2f} s; accuracy {acc:.2f} on 100 held-out tickets; {per * 1000:.2f} ms a ticket")
print("on the same 12:", int((clf.predict(sample.text) == sample.department).sum()), "right")

In [ ]:
slm.save_json("12_02_prompts.json", {
    "raw_reply": raw_reply, "chat_reply": chat_reply, "template_tokens": template_tokens,
    "labels": labels, "logits": L, "raw_predictions": raw_pred, "calibrated_predictions": cal_pred,
    "classifier_accuracy": acc, "classifier_ms_per_ticket": per * 1000})
check_12_02()

For a fixed job with labelled examples, the trained classifier wins on every axis here: it is more
accurate, it is tens of thousands of times faster, and it has no prior to calibrate away. A prompt is
how you start when you have no labels, or when the job changes every week; large hosted models route far
better than this 360-million-weight one, but they still cost a network call per ticket.

## 7. Exit ticket

**x2.** Why did subtracting each department's mean logit help? (a) it removed a preference for some
department names that was added to every ticket, (b) it made the model read the tickets again,
(c) it trained the model on the 12 tickets

In [ ]:
ask("x2", "")

Explain it back: the raw transcript and the chat template went into the same weights. Why did one produce
a single reply and the other both sides of the conversation?

*Your explanation:* 